# Tesseract fine-tune: bilingual enterprise print

Train and evaluate Tesseract independently from VietOCR on the same corrected bilingual enterprise printed rows and source-PDF-disjoint splits.

## 1. Install Tesseract and data dependencies

This kernel uses Kaggle CPU. Every crop is converted to PNG because tesstrain only defines LSTM generation rules for PNG/TIFF images. Ground-truth preparation reports progress every 500 samples.

In [ ]:
import importlib.util, subprocess, sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run([
    "apt-get", "install", "-y", "-qq", "git", "make", "pkg-config",
    "tesseract-ocr", "tesseract-ocr-vie", "libtesseract-dev",
    "libleptonica-dev", "fonts-dejavu-core",
], check=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "vietocr==0.3.13"])
for module, package in (("imgaug","imgaug==0.4.0"),("prefetch_generator","prefetch_generator==1.0.1"),
    ("einops","einops==0.2.0"),("albumentations","albumentations==1.4.2"),("lmdb","lmdb"),
    ("gdown","gdown==4.4.0"),("rapidfuzz","rapidfuzz")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])

import numpy as np
# Compatibility for imgaug 0.4 on Kaggle's NumPy 2 / Python 3.13 image.
if not hasattr(np,"sctypes"):
    np.sctypes={"int":[np.int8,np.int16,np.int32,np.int64],"uint":[np.uint8,np.uint16,np.uint32,np.uint64],
        "float":[np.float16,np.float32,np.float64],"complex":[np.complex64,np.complex128],
        "others":[bool,object,bytes,str]}
print(subprocess.check_output(["tesseract","--version"],text=True).splitlines()[0])


## 2. Load and validate the shared split

Use corrected Vietnamese and English enterprise printed rows from the shared source-PDF-disjoint dataset.


In [ ]:
import hashlib, json, random, shutil, unicodedata, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
from rapidfuzz.distance import Levenshtein
from vietocr.tool.config import Cfg

INPUT_ROOT=Path('/kaggle/input'); PACKAGE=next(INPUT_ROOT.rglob('dataset-info.json')).parent
WORK=Path('/kaggle/working/enterprise-printed-ocr'); WORK.mkdir(parents=True,exist_ok=True)
DATA=WORK/'data'; DATA.mkdir(parents=True,exist_ok=True)
image_dir=PACKAGE/'images'
image_zip=next(PACKAGE.glob('images*.zip'),None)
if not (DATA/'images').exists():
    if image_dir.is_dir():
        (DATA/'images').symlink_to(image_dir,target_is_directory=True)
    elif image_zip:
        with zipfile.ZipFile(image_zip) as zf: zf.extractall(DATA)
    else:
        raise FileNotFoundError('Combined dataset has neither images/ nor images.zip')
for name in ('dataset-info.json','annotation_train.txt','annotation_validation.txt','annotation_test.txt','manifest_train.jsonl','manifest_validation.jsonl','manifest_test.jsonl'):
    shutil.copy2(PACKAGE/name,DATA/name)
info=json.loads((DATA/'dataset-info.json').read_text(encoding='utf-8'))

def read_split(name):
    annotations=(DATA/f'annotation_{name}.txt').read_text(encoding='utf-8').splitlines()
    manifest=[json.loads(line) for line in (DATA/f'manifest_{name}.jsonl').read_text(encoding='utf-8').splitlines()]
    if len(annotations)!=len(manifest): raise ValueError(f'{name}: annotation/manifest count mismatch')
    rows=[]
    for line,record in zip(annotations,manifest):
        image,sep,text=line.partition('\t')
        if not sep or image!=record['image'] or text!=record['text']: raise ValueError(f'Bad {name} row: {line[:100]}')
        relative=Path(image)
        if relative.is_absolute() or '..' in relative.parts: raise ValueError(f'Unsafe image path: {image}')
        path=DATA/relative
        if not path.is_file() or not text.strip(): raise ValueError(f'Missing image or empty label: {image}')
        record=dict(record); record.update(path=path,text=text); rows.append(record)
    return rows
train_all=read_split('train'); val_all=read_split('validation'); test_all=read_split('test')
correction_counts={split:sum(bool(row.get('label_correction_applied')) for row in rows)
    for split,rows in [('train',train_all),('validation',val_all),('test',test_all)]}
declared_corrections=info.get('confirmed_transcript_corrections_applied',{})
if correction_counts!=declared_corrections:
    print(json.dumps({'warning':'dataset-info correction counts are stale; manifest counts will be used',
        'dataset_info':declared_corrections,'manifest':correction_counts},ensure_ascii=False),flush=True)
    info['confirmed_transcript_corrections_applied']=correction_counts
print('Verified materialized dataset corrections:',correction_counts,flush=True)
def group_set(rows,key): return {row[key] for row in rows if row.get(key) is not None}
for key in ('source_pdf_sha256',):
    assert not group_set(train_all,key)&group_set(val_all,key)
    assert not group_set(train_all,key)&group_set(test_all,key)
    assert not group_set(val_all,key)&group_set(test_all,key)
assert len(test_all)==info['counts']['test']['rows']
printed_domains={'enterprise_printed','enterprise_english'}
for split,rows in [('train',train_all),('validation',val_all),('test',test_all)]:
    unexpected={row['domain'] for row in rows}-printed_domains
    if unexpected: raise ValueError(f'Unexpected domains in {split}: {sorted(unexpected)}')
print(json.dumps({'experiment_domains':sorted(printed_domains),
    'printed_rows_before_vocab_filter':{'train':len(train_all),'validation':len(val_all),'test':len(test_all)}},
    ensure_ascii=False,indent=2),flush=True)


## 3. Metrics

In [ ]:
PUNCTUATION_TRANSLATION = str.maketrans({
    '‘': "'", '’': "'", '‚': "'", '‛': "'",
    '“': '"', '”': '"', '„': '"', '‟': '"', '»': '"',
    '–': '-', '—': '-', '−': '-', '•': '*', '™': 'TM', '›': '>', '⁄': '/',
})
def normalize_whitespace(text):
    return ' '.join(unicodedata.normalize('NFC',str(text)).strip().split())
def normalize(text):
    return normalize_whitespace(str(text).translate(PUNCTUATION_TRANSLATION))
def metric_pair(refs,hyps):
    char_errors=sum(Levenshtein.distance(a,b) for a,b in zip(refs,hyps))
    word_errors=sum(Levenshtein.distance(a.split(),b.split()) for a,b in zip(refs,hyps))
    return {'CER':char_errors/max(sum(map(len,refs)),1),
        'WER':word_errors/max(sum(len(a.split()) for a in refs),1),
        'exact_match':sum(a==b for a,b in zip(refs,hyps))/max(len(refs),1)}
def score(name,rows,predictions):
    refs=[normalize(row['text']) for row in rows]; hyps=[normalize(text) for text in predictions]
    raw_refs=[normalize_whitespace(row.get('reference_raw',row['text'])) for row in rows]
    raw_hyps=[normalize_whitespace(text) for text in predictions]
    normalized_metrics=metric_pair(refs,hyps); raw_metrics=metric_pair(raw_refs,raw_hyps)
    return {'system':name,'n_lines':len(refs),**normalized_metrics,
        'raw_CER':raw_metrics['CER'],'raw_WER':raw_metrics['WER'],'raw_exact_match':raw_metrics['exact_match']}
def metrics_by_domain(name,rows,predictions):
    result=[score(name+' / all',rows,predictions)]
    for domain in ('enterprise_printed','enterprise_english'):
        indices=[i for i,row in enumerate(rows) if row['domain']==domain]
        result.append(score(name+' / '+domain,[rows[i] for i in indices],[predictions[i] for i in indices]))
    return result


## 4. Filter the shared character set and prepare Tesseract PNG ground truth

In [ ]:
base_config=Cfg.load_config_from_name('vgg_seq2seq')
model_vocab=set(base_config['vocab'])
def supported_rows(rows):
    kept=[]; excluded=[]
    for original in rows:
        row={**original,'reference_raw':original['text'],'text':normalize(original['text'])}
        (kept if set(row['text'])<=model_vocab else excluded).append(row)
    return kept,excluded
train_rows,train_excluded=supported_rows(train_all)
val_rows,val_excluded=supported_rows(val_all)
test_rows,test_excluded=supported_rows(test_all)
print({'train_used':len(train_rows),'train_oov_excluded':len(train_excluded),'validation_used':len(val_rows),
       'validation_oov_excluded':len(val_excluded),'test_used':len(test_rows),'test_oov_excluded':len(test_excluded)},flush=True)

GT=WORK/'ground-truth'; GT.mkdir(parents=True,exist_ok=True)
for i,row in enumerate(train_rows):
    stem=f'line-{i:05d}'
    with Image.open(row['path']) as image:
        image.convert('RGB').save(GT/f'{stem}.png',format='PNG')
    (GT/f'{stem}.gt.txt').write_text(row['text'],encoding='utf-8')
    if (i+1)%500==0 or i+1==len(train_rows):
        print(f'Prepared Tesseract ground truth: {i+1}/{len(train_rows)}',flush=True)
assert len(list(GT.glob('*.png')))==len(train_rows)
assert len(list(GT.glob('*.gt.txt')))==len(train_rows)


## 5. Fine-tune Tesseract LSTM

Continue from `vie` with the selected 5,000-iteration setting. Training output streams into this cell.

In [ ]:
import os
TESSTRAIN=WORK/'tesstrain'
subprocess.run(['git','clone','--depth','1','https://github.com/tesseract-ocr/tesstrain.git',str(TESSTRAIN)],check=True)
TESSDATA=WORK/'tessdata'; TESSDATA.mkdir(exist_ok=True)
subprocess.run(['wget','-q','-O',str(TESSDATA/'vie.traineddata'),'https://raw.githubusercontent.com/tesseract-ocr/tessdata_best/main/vie.traineddata'],check=True)
train_cmd=['make','training','MODEL_NAME=vie_enterprise_print','START_MODEL=vie',f'TESSDATA={TESSDATA}',
    f'GROUND_TRUTH_DIR={GT}','MAX_ITERATIONS=5000','RATIO_TRAIN=0.9']
print(f'Starting Tesseract LSTM training on {len(train_rows)} PNG crops (5000 iterations)',flush=True)
training_log=WORK/'tesstrain-training.log'
with training_log.open('w',encoding='utf-8') as log:
    train_result=subprocess.run(train_cmd,cwd=TESSTRAIN,check=False,stdout=log,stderr=subprocess.STDOUT,
        text=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})
if train_result.returncode:
    tail=training_log.read_text(encoding='utf-8',errors='replace').splitlines()[-80:]
    raise RuntimeError(f'Tesseract training failed ({train_result.returncode}); last log lines:\n'+'\n'.join(tail))
print('Tesseract training log saved:',training_log,flush=True)
candidates=list(TESSTRAIN.rglob('vie_enterprise_print.traineddata'))
if not candidates: raise FileNotFoundError('tesstrain produced no vie_enterprise_print.traineddata')
TESSERACT_FT=WORK/'vie_enterprise_print.traineddata'
shutil.copy2(max(candidates,key=lambda path:path.stat().st_size),TESSERACT_FT)
print('Tesseract model:',TESSERACT_FT,'bytes:',TESSERACT_FT.stat().st_size,flush=True)


## 6. Evaluate pretrained and fine-tuned Tesseract

Report validation and held-out test metrics for Vietnamese and English printed rows, separately and together.

In [ ]:
def predict_tesseract(rows,model_name):
    output=[]
    for i,row in enumerate(rows):
        command=['tesseract',str(row['path']),'stdout','-l',model_name,'--psm','7','--tessdata-dir',str(TESSDATA)]
        result=subprocess.run(command,capture_output=True,text=True)
        if result.returncode: raise RuntimeError(f'Tesseract {model_name} failed: {result.stderr[-1200:]}')
        output.append(result.stdout.strip())
        if (i+1)%200==0 or i+1==len(rows): print(f'{model_name} predictions: {i+1}/{len(rows)}',flush=True)
    return output
(TESSDATA/'vie_enterprise_print.traineddata').write_bytes(TESSERACT_FT.read_bytes())
tess_base_preds=predict_tesseract(test_rows,'vie')
tess_ft_preds=predict_tesseract(test_rows,'vie_enterprise_print')
validation_preds=predict_tesseract(val_rows,'vie_enterprise_print')
tesseract_results=metrics_by_domain('Tesseract pretrained',test_rows,tess_base_preds)
tesseract_results+=metrics_by_domain('Tesseract fine-tuned',test_rows,tess_ft_preds)
validation_results=metrics_by_domain('Tesseract fine-tuned / validation',val_rows,validation_preds)
print(json.dumps({'validation':validation_results,'test':tesseract_results},ensure_ascii=False,indent=2),flush=True)


## 7. Save and upload Tesseract artifacts

Upload only the trained model and aggregate metrics; no crops, labels, or line predictions are sent to Hugging Face.

In [ ]:
import hashlib

metrics={'dataset':info,'label_corrections_applied':info.get('confirmed_transcript_corrections_applied',{}),'text_normalization':'NFC; collapse Unicode whitespace; canonicalize curly quotes and dash variants to ASCII; map bullets to *, trademark to TM, and fraction slash to /','experiment_scope':'enterprise printed only; Vietnamese and English','start_model':'vie','max_iterations':5000,'ratio_train':0.9,
    'train_rows_used':len(train_rows),'train_rows_oov_excluded':len(train_excluded),
    'validation_rows_used':len(val_rows),'validation_rows_oov_excluded':len(val_excluded),
    'test_rows_used':len(test_rows),'test_rows_oov_excluded':len(test_excluded),
    'validation_metrics':validation_results,'test_metrics':tesseract_results}
metrics_path=WORK/'tesseract-metrics.json'
metrics_path.write_text(json.dumps(metrics,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
metadata_path=WORK/'tesseract-run-metadata.json'
metadata_path.write_text(json.dumps({'dataset_id':'Enterprise Printed OCR Lines',
    'weights_sha256':hashlib.sha256(TESSERACT_FT.read_bytes()).hexdigest(),'metrics':tesseract_results},
    ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Tesseract model and metrics saved:',TESSERACT_FT,metrics_path,flush=True)
